# Gold Annotation Reconciliation and Pipeline Comparison (Dict 4)

This notebook performs the post-annotation analysis for dict 4 (the first dict
to complete annotation). It runs in three stages:

## Stage 1 Reconcile A1 and A2 submissions 

Both annotators worked the same 100 source lemmas independently. For each
lemma, we compare A1's and A2's submissions row-by-row and produce parcor C
following the strict-intersection rule from §7.1 of the Gold Annotation Plan:

- **Both extracted, content matches**: included in parcor C
- **Both marked empty** (no example sentences): recorded as agreed-no-pair
- **Both marked not found**: recorded as agreed-not-found
- **All other cases**: disagreement, logged for researcher review, excluded from C

We compute Cohen's κ on the extract/skip decision per the plan §7.2.

## Stage 2 Compare parcor C against the legacy pipeline (parcor A)

For each lemma in C, look up the legacy pipeline's output (from
`<dict>_Parcor_audit.csv`) and classify the comparison outcome:

| C status | A status | Outcome |
|---|---|---|
| Has pair | Has matching pair | TP — pipeline correctly extracted |
| Has pair | Has different content | wrong_extraction — pipeline extracted wrong content |
| Has pair | No row for this lemma | missed_lemma — pipeline didn't extract anything |
| Empty | Has pair | FP — pipeline extracted from a lemma with no example |
| Empty | No row | TN — both agree there's nothing |

Match levels reported: **strict exact**, **normalized** (lowercase, whitespace-collapsed,
punctuation-stripped), and **fuzzy** (character n-gram similarity ≥ 0.85).

## Stage 3 POS comparison

The annotator added a `POS` column to each row. The morphology file
(`<dict>_Morphology.csv`) also has POS data via `form → tag` lookup. We compare:

- **A1 vs A2 POS agreement** (primary)
- **Annotator POS vs morphology POS** (secondary, only where morphology has a match)

Coverage is expected to be partial — the morphology file may not have every
sampled lemma.

## Outputs (in `../csvAnalysis/gold_extraction/comparison/`)

- `4_parcor_C.csv`: reconciled gold (strict intersection)
- `4_disagreement.csv`: rows excluded from C, for researcher review
- `4_iaa_report.csv`: per-dict κ and agreement metrics
- `4_comparison_A_vs_C.csv`: row-by-row pipeline-vs-gold comparison
- `4_comparison_summary.csv`: aggregated precision/recall/F1

## 1. Configuration

In [454]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from collections import Counter

# ===========================================================================
# RUN CONFIGURATION — edit these four variables, then run all cells
# ===========================================================================
DICT_ID  = "91"        # one of: "4", "19", "24", "46", "91"
ALGO     = "new"   # "legacy" or "new"
STAGE    = "fixed"    # "fixed" or "spellchecked"
STRATEGY = None    # None when STAGE="fixed"; "original" | "B" | "C" when "spellchecked"
# ===========================================================================

# Validate config
NEW_GOLD_DICTS = ["4", "19", "24", "46", "91"]
assert DICT_ID in NEW_GOLD_DICTS, f"DICT_ID must be one of {NEW_GOLD_DICTS}, got {DICT_ID!r}"
assert ALGO in ("legacy", "new"), f"ALGO must be 'legacy' or 'new', got {ALGO!r}"
assert STAGE in ("fixed", "spellchecked"), f"STAGE must be 'fixed' or 'spellchecked', got {STAGE!r}"
if STAGE == "fixed":
    assert STRATEGY is None, "STRATEGY must be None when STAGE='fixed'"
else:
    assert STRATEGY in ("original", "B", "C"), \
        f"STRATEGY must be 'original'|'B'|'C' when STAGE='spellchecked', got {STRATEGY!r}"

# === Annotator files (both for this dict) ===
GOLD_DIR = Path("../csvAnalysis/gold_extraction")
A1_PATH = GOLD_DIR / f"A1_{DICT_ID}_extraction.csv"
A2_PATH = GOLD_DIR / f"A5_{DICT_ID}_extraction.csv"

# === Resolve pipeline parcor input from config ===
EKSTRAKSI_ROOT = Path("../Ekstraksi")
_NEW_INFIX = " (New)" if ALGO == "new" else ""

if STAGE == "fixed":
    PARCOR_DIR = EKSTRAKSI_ROOT / f"11.{_NEW_INFIX} Parallel Corpus - Fixed"
    PARCOR_PATH = PARCOR_DIR / f"{DICT_ID}_Parcor_audit.csv"
else:
    PARCOR_DIR = EKSTRAKSI_ROOT / f"12.{_NEW_INFIX} Parallel Corpus - Spellcheck Detection" / f"dict_strategy_{STRATEGY}"
    PARCOR_PATH = PARCOR_DIR / f"{DICT_ID}_Parcor_spellcheck.csv"

# Morphology lookup (for POS comparison) — algo-dependent but stage-independent
# Both Fixed and Spellchecked use the Fixed morphology since POS doesn't change with spellcheck
MORPH_BASE = EKSTRAKSI_ROOT / f"10.{_NEW_INFIX} Morphology - Fixed"
MORPH_CANDIDATES = [
    MORPH_BASE / f"{DICT_ID}_Morphology.csv",
    EKSTRAKSI_ROOT / f"{DICT_ID}_Morphology.csv",  # legacy fallback
]

# === Match thresholds ===
FUZZY_THRESHOLD = 0.75  # character n-gram cosine for fuzzy match
                        # Lowered from 0.85 — at 0.85, fuzzy adds 0-1 TPs over normalized
                        # across test dicts. 0.75 makes fuzzy a distinct sensitivity column.

# === Concatenation detection thresholds ===
CONCAT_LENGTH_RATIO = 2.0   # A's content is >2x as long as C's
CONCAT_MIN_SENTENCE_ENDS = 2  # A's content has 2+ sentence-final marks

# === Output directory derived from config ===
_stage_label = "fixed" if STAGE == "fixed" else f"spellcheck_{STRATEGY}"
_condition = f"{ALGO}_algo_{_stage_label}"
OUT_DIR = Path("../evaluationCSV/evaluation_new_gold_1") / _condition
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== Configuration ===")
print(f"  DICT_ID:  {DICT_ID}")
print(f"  ALGO:     {ALGO}")
print(f"  STAGE:    {STAGE}")
print(f"  STRATEGY: {STRATEGY}")
print(f"\n=== Resolved paths ===")
print(f"  A1:       {A1_PATH}")
print(f"  A2:       {A2_PATH}")
print(f"  Parcor:   {PARCOR_PATH}")
print(f"  OUT_DIR:  {OUT_DIR.resolve()}")

=== Configuration ===
  DICT_ID:  91
  ALGO:     new
  STAGE:    fixed
  STRATEGY: None

=== Resolved paths ===
  A1:       ..\csvAnalysis\gold_extraction\A1_91_extraction.csv
  A2:       ..\csvAnalysis\gold_extraction\A5_91_extraction.csv
  Parcor:   ..\Ekstraksi\11. (New) Parallel Corpus - Fixed\91_Parcor_audit.csv
  OUT_DIR:  C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\evaluationCSV\evaluation_new_gold_1\new_algo_fixed


## 2. Load annotator files

In [455]:
def load_annotator(path):
    if not path.exists():
        raise FileNotFoundError(f"Annotator file not found: {path}")
    df = pd.read_csv(path)
    # Normalize string columns: fill NaN, cast to str, strip, and collapse
    # any embedded CR/LF (Windows line endings inside CSV cells become \r\n
    # which would otherwise cause strict-match failures on otherwise-identical content)
    for col in ["source_lemma", "main_lemma", "POS", "kalimat_asal", "kalimat_tujuan", "notes"]:
        if col in df.columns:
            df[col] = (df[col].fillna("").astype(str)
                       .str.replace("\r", " ", regex=False)
                       .str.replace("\n", " ", regex=False)
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip())
    return df


a1 = load_annotator(A1_PATH)
a2 = load_annotator(A2_PATH)
print(f"A1: {len(a1)} rows")
print(f"A2: {len(a2)} rows")

# Verify both have the same source_lemmas (otherwise reconciliation row-by-row breaks)
a1_lemmas = set(a1["source_lemma"])
a2_lemmas = set(a2["source_lemma"])
if a1_lemmas != a2_lemmas:
    only_a1 = a1_lemmas - a2_lemmas
    only_a2 = a2_lemmas - a1_lemmas
    print(f"⚠ Lemma sets differ: {len(only_a1)} only in A1, {len(only_a2)} only in A2")
    if only_a1:
        print(f"   Only in A1: {sorted(only_a1)[:5]}")
    if only_a2:
        print(f"   Only in A2: {sorted(only_a2)[:5]}")
else:
    print(f"✓ Both annotators have the same {len(a1_lemmas)} source_lemmas")

A1: 100 rows
A2: 100 rows
✓ Both annotators have the same 100 source_lemmas


## 3. Normalization helpers for matching

We compute three match levels:

- **Strict**: exact byte equality (after the load-time `.strip()`).
- **Normalized**: lowercased, whitespace collapsed to single spaces, leading/trailing
  punctuation stripped, internal `,;.?!` preserved.
- **Fuzzy**: character n-gram (3-gram) Jaccard similarity ≥ FUZZY_THRESHOLD.

In [456]:
_normalize_re = re.compile(r"\s+")

def normalize_text(s: str) -> str:
    """Lowercase, collapse internal whitespace, strip leading/trailing punctuation."""
    if not s:
        return ""
    s = s.lower().strip()
    s = _normalize_re.sub(" ", s)
    # Strip leading/trailing punctuation but preserve interior
    s = s.strip(" .,;:!?\"'()[]{}")
    return s


def char_ngrams(s: str, n: int = 3) -> set:
    """Character n-grams from a normalized string."""
    s_norm = normalize_text(s)
    if len(s_norm) < n:
        return {s_norm} if s_norm else set()
    return {s_norm[i:i+n] for i in range(len(s_norm) - n + 1)}


def fuzzy_similarity(a: str, b: str) -> float:
    """Jaccard similarity on character 3-grams."""
    ga = char_ngrams(a)
    gb = char_ngrams(b)
    if not ga and not gb:
        return 1.0
    if not ga or not gb:
        return 0.0
    return len(ga & gb) / len(ga | gb)


def match_level(c_text: str, a_text: str) -> dict:
    """Compute strict/normalized/fuzzy match between two text strings."""
    strict = (c_text == a_text)
    normalized = (normalize_text(c_text) == normalize_text(a_text))
    fuzzy_score = fuzzy_similarity(c_text, a_text)
    fuzzy = fuzzy_score >= FUZZY_THRESHOLD
    return {
        "strict": strict,
        "normalized": normalized,
        "fuzzy": fuzzy,
        "fuzzy_score": round(fuzzy_score, 3),
    }


# Smoke test
print("Smoke tests:")
print(f"  identical: {match_level('hello world', 'hello world')}")
print(f"  case diff: {match_level('Hello World', 'hello world')}")
print(f"  extra space: {match_level('dilema  yg', 'dilema yg')}")
print(f"  typo: {match_level('memuat', 'membuat')}")

Smoke tests:
  identical: {'strict': True, 'normalized': True, 'fuzzy': True, 'fuzzy_score': 1.0}
  case diff: {'strict': False, 'normalized': True, 'fuzzy': True, 'fuzzy_score': 1.0}
  extra space: {'strict': False, 'normalized': True, 'fuzzy': True, 'fuzzy_score': 1.0}
  typo: {'strict': False, 'normalized': False, 'fuzzy': False, 'fuzzy_score': 0.286}


## 4. Stage 1 Reconcile A1 and A2: parcor C

For each `source_lemma`, classify the row outcome:

- `agreed_extracted_match`: both filled, content matches at the normalized level
- `agreed_no_pair`: both empty with no-example-sentences notes
- `agreed_not_found`: both empty with lemma-not-found notes
- `disagree_content`: both filled but content differs at normalized level
- `disagree_extract`: one filled, the other empty
- `unclear`: both empty but no notes (annotator-incomplete)

Only `agreed_extracted_match` enters parcor C. The plan specified strict
exact match for the notes convention, but we accept either of the documented
strings (`"no example sentences"`, `"no parcor in entry"`, `"lemma not found in PDF"`,
or close variants).

In [457]:
# Semantic families for empty-row notes.
# Both annotators may use different phrasings for the same intent.
# We classify by keyword groups rather than exact pattern matching.
NO_PAIR_KEYWORDS = [
    "example",      # "no example sentences", "no example found", "no example", "example tidak ada"
    "contoh",       # "tidak ada contoh", "contoh kalimat tidak ada"
    "no parcor",
    "no pair",
]
NOT_FOUND_KEYWORDS = [
    "not found",    # "lemma not found in PDF", "no entry found", "not found in pdf"
    "no entry",
    "tidak ditemukan",
    "tidak ada di pdf",
    "tidak ada entri",
]


def classify_notes(notes: str) -> str:
    """Classify the notes string into one of: 'no_pair', 'not_found', 'other', or ''.

    Uses keyword-family matching to handle paraphrased annotator notes
    (e.g. 'no example sentences' and 'no example found' both classify as 'no_pair').
    Order matters: 'not_found' checked first because it's more specific.
    """
    if not notes:
        return ""
    n = notes.lower().strip()
    for kw in NOT_FOUND_KEYWORDS:
        if kw in n:
            return "not_found"
    for kw in NO_PAIR_KEYWORDS:
        if kw in n:
            return "no_pair"
    return "other"


def row_has_pair(row) -> bool:
    """A row has a pair if kalimat_asal AND kalimat_tujuan are both non-empty."""
    return bool(row["kalimat_asal"]) and bool(row["kalimat_tujuan"])


# Build aligned dataframe (one row per source_lemma, with A1 and A2 columns side-by-side)
common_lemmas = sorted(set(a1["source_lemma"]) & set(a2["source_lemma"]))

reconcile_rows = []
for lemma in common_lemmas:
    r1 = a1[a1["source_lemma"] == lemma].iloc[0]
    r2 = a2[a2["source_lemma"] == lemma].iloc[0]

    r1_has = row_has_pair(r1)
    r2_has = row_has_pair(r2)
    r1_notes = classify_notes(r1["notes"])
    r2_notes = classify_notes(r2["notes"])

    if r1_has and r2_has:
        # Both extracted — compare content
        asal_match = match_level(r1["kalimat_asal"], r2["kalimat_asal"])
        tuj_match  = match_level(r1["kalimat_tujuan"], r2["kalimat_tujuan"])
        if asal_match["normalized"] and tuj_match["normalized"]:
            outcome = "agreed_extracted_match"
        else:
            outcome = "disagree_content"
    elif not r1_has and not r2_has:
        # Both empty — classify by notes family
        if r1_notes == "no_pair" and r2_notes == "no_pair":
            outcome = "agreed_no_pair"
        elif r1_notes == "not_found" and r2_notes == "not_found":
            outcome = "agreed_not_found"
        elif r1_notes == r2_notes and r1_notes:
            # Both have non-empty notes that didn't match either family but matched each other
            outcome = f"agreed_{r1_notes}"
        elif r1_notes and r2_notes:
            # Both have notes but they classify into different families
            # (e.g., A1 says "no example" but A2 says "lemma not found")
            # Both still agreed to skip — count as agreed_empty_other rather than unclear
            outcome = "agreed_empty_other"
        else:
            # At least one has no notes at all — genuinely unclear
            outcome = "unclear"
    else:
        outcome = "disagree_extract"

    reconcile_rows.append({
        "source_lemma":      lemma,
        "main_lemma_a1":     r1["main_lemma"],
        "main_lemma_a2":     r2["main_lemma"],
        "pos_a1":            r1["POS"],
        "pos_a2":            r2["POS"],
        "kalimat_asal_a1":   r1["kalimat_asal"],
        "kalimat_asal_a2":   r2["kalimat_asal"],
        "kalimat_tujuan_a1": r1["kalimat_tujuan"],
        "kalimat_tujuan_a2": r2["kalimat_tujuan"],
        "notes_a1":          r1["notes"],
        "notes_a2":          r2["notes"],
        "notes_family_a1":   r1_notes,
        "notes_family_a2":   r2_notes,
        "outcome":           outcome,
    })

reconcile_df = pd.DataFrame(reconcile_rows)

# Outcome breakdown
outcome_counts = Counter(reconcile_df["outcome"])
print("=== Reconciliation outcomes ===")
for outcome, n in outcome_counts.most_common():
    print(f"  {outcome}: {n}")
print(f"\n  Total: {len(reconcile_df)}")

=== Reconciliation outcomes ===
  agreed_extracted_match: 88
  disagree_content: 9
  agreed_no_pair: 3

  Total: 100


## 5. Build parcor C and disagreement file

Parcor C contains only `agreed_extracted_match` rows. For those, we record:
- Annotator-agreed content (`kalimat_asal`, `kalimat_tujuan`, taking A1's version
  since they match at normalized level; the strict-level difference, if any, is
  preserved in the disagreement file even though the row counts as agreed).
- Agreed POS (if A1 and A2's POS strings match, otherwise marked).

Disagreement file contains every row that isn't `agreed_extracted_match` or the
two `agreed_no_pair`/`agreed_not_found` empty cases, i.e., the rows where the
researcher needs to make a call.

In [458]:
# Parcor C: agreed-extracted-match rows
c_rows = reconcile_df[reconcile_df["outcome"] == "agreed_extracted_match"].copy()
parcor_C = pd.DataFrame({
    "dict_id":        DICT_ID,
    "source_lemma":   c_rows["source_lemma"],
    "main_lemma":     c_rows["main_lemma_a1"],  # they normally agree
    "kalimat_asal":   c_rows["kalimat_asal_a1"],
    "kalimat_tujuan": c_rows["kalimat_tujuan_a1"],
    "pos":            c_rows.apply(
        lambda r: r["pos_a1"] if r["pos_a1"] == r["pos_a2"] else f"{r['pos_a1']}|{r['pos_a2']}",
        axis=1
    ),
})
parcor_C_path = OUT_DIR / f"{DICT_ID}_parcor_C.csv"
parcor_C.to_csv(parcor_C_path, index=False)
print(f"Parcor C: {len(parcor_C)} agreed pairs → {parcor_C_path.name}")

# Disagreement file: only true disagreements (not agreed_empty_other or agreed_*)
disagreement = reconcile_df[
    reconcile_df["outcome"].isin(["disagree_content", "disagree_extract", "unclear"])
].copy()
disagreement_path = OUT_DIR / f"{DICT_ID}_disagreement.csv"
disagreement.to_csv(disagreement_path, index=False)
print(f"Disagreement: {len(disagreement)} rows → {disagreement_path.name}")

# Quick peek
print("\nDisagreement breakdown:")
for outcome, n in Counter(disagreement["outcome"]).most_common():
    print(f"  {outcome}: {n}")

Parcor C: 88 agreed pairs → 91_parcor_C.csv
Disagreement: 9 rows → 91_disagreement.csv

Disagreement breakdown:
  disagree_content: 9


## 6. Compute IAA metrics

We report multiple agreement metrics because **Cohen's κ alone is brittle when
the base rate is highly skewed** (e.g., when 96% of decisions are "extract,"
even small disagreements drive κ near zero or negative). The plan still
prioritizes κ, but we report it in three variants and pair it with chance-
robust alternatives.

**Agreement statistics:**

- **Raw agreement rate**: fraction of lemmas where both annotators agreed (any agreement category, including agreed-extracted-and-matching, agreed-no-pair, agreed-not-found, and agreed-empty-other). Most direct measure.
- **Percent agreement on extract/skip**: fraction agreeing on the binary "did the annotator extract a pair?" question.
- **Conditional content agreement**: among rows where both annotators extracted, the fraction where content matches at normalized level.

**Chance-corrected metrics (in priority order):**

- **Cohen's κ (extract/skip)**: the original metric from the plan. Binary: did annotator extract? Known brittleness with high base rates.
- **Cohen's κ (3-class outcome)**: extends to 3 categories: `extracted` / `no_pair` / `not_found`. Reduces base rate skew when the distribution actually has 3 categories.
- **Cohen's κ (content-given-extracted)**: only computed on rows where both annotators extracted. Treats "content matches" as binary (yes/no). Measures content-level agreement independent of the extract/skip imbalance.
- **Gwet's AC1 (extract/skip)**: chance-corrected statistic robust to base-rate skew. Recommended for high-agreement annotation tasks.

In [459]:
from sklearn.metrics import cohen_kappa_score

n_total = len(reconcile_df)
n_agreed_extracted   = outcome_counts.get("agreed_extracted_match", 0)
n_agreed_no_pair     = outcome_counts.get("agreed_no_pair", 0)
n_agreed_not_found   = outcome_counts.get("agreed_not_found", 0)
n_agreed_empty_other = outcome_counts.get("agreed_empty_other", 0)
n_disagree_content   = outcome_counts.get("disagree_content", 0)
n_disagree_extract   = outcome_counts.get("disagree_extract", 0)
n_unclear            = outcome_counts.get("unclear", 0)

# Raw agreement (any agreement category)
total_agreed = (n_agreed_extracted + n_agreed_no_pair + n_agreed_not_found
                + n_agreed_empty_other)
row_agreement_rate = total_agreed / n_total

# Conditional content agreement (when both extracted)
cond_content_agreement = (
    n_agreed_extracted / (n_agreed_extracted + n_disagree_content)
    if (n_agreed_extracted + n_disagree_content) else float("nan")
)


# Build per-row label vectors for κ computation
def extract_binary(row, anno):
    """1 if annotator extracted a pair, 0 otherwise."""
    return 1 if (row[f"kalimat_asal_{anno}"] and row[f"kalimat_tujuan_{anno}"]) else 0


def extract_3class(row, anno):
    """3-class label: 'extracted', 'no_pair', or 'not_found'.
    Uses the notes family to distinguish empty cases.
    """
    if row[f"kalimat_asal_{anno}"] and row[f"kalimat_tujuan_{anno}"]:
        return "extracted"
    family = row[f"notes_family_{anno}"]
    if family == "not_found":
        return "not_found"
    # 'no_pair', 'other', or '' all fall into 'no_pair' for kappa purposes
    return "no_pair"


extract_a1_bin = reconcile_df.apply(lambda r: extract_binary(r, "a1"), axis=1)
extract_a2_bin = reconcile_df.apply(lambda r: extract_binary(r, "a2"), axis=1)
extract_a1_3c = reconcile_df.apply(lambda r: extract_3class(r, "a1"), axis=1)
extract_a2_3c = reconcile_df.apply(lambda r: extract_3class(r, "a2"), axis=1)

# Percent agreement on extract/skip (binary)
percent_agreement_binary = (extract_a1_bin == extract_a2_bin).mean()

# === Cohen's κ — three variants ===
kappa_binary = cohen_kappa_score(extract_a1_bin, extract_a2_bin)
kappa_3class = cohen_kappa_score(extract_a1_3c, extract_a2_3c)

# Content-given-extracted κ: only on rows where both extracted, binary match/no-match
both_extracted_mask = reconcile_df["outcome"].isin(["agreed_extracted_match", "disagree_content"])
both_extracted_df = reconcile_df[both_extracted_mask]
if len(both_extracted_df) >= 2:
    content_match_a1 = both_extracted_df.apply(
        lambda r: 1 if r["outcome"] == "agreed_extracted_match" else 0, axis=1
    )
    # For κ to be meaningful we need annotator vs annotator, but reconciliation collapses
    # this to a single yes/no per row. Use the symmetric form: both annotators "voted"
    # the same outcome by definition (agreed_extracted_match means yes-yes,
    # disagree_content means no-no since they disagree on what to write).
    # This makes content-κ degenerate. Use percent-agreement on content instead:
    content_pct_agreement = (both_extracted_df["outcome"] == "agreed_extracted_match").mean()
    kappa_content = float("nan")  # not meaningful for our 1-rater-per-side setup
else:
    content_pct_agreement = float("nan")
    kappa_content = float("nan")


# === Gwet's AC1 — chance-robust alternative to κ on extract/skip ===
def gwets_ac1(rater1, rater2):
    """Gwet's AC1: chance-corrected agreement robust to base-rate skew.
    
    AC1 = (Po - Pe) / (1 - Pe)
    where Po is observed agreement and Pe is chance agreement computed from
    the *average* category proportions across both raters (rather than from
    each rater's marginals, as in κ).
    """
    r1 = list(rater1)
    r2 = list(rater2)
    n = len(r1)
    if n == 0:
        return float("nan")
    # Observed agreement
    po = sum(1 for a, b in zip(r1, r2) if a == b) / n
    # Chance agreement: average category proportion across both raters,
    # then sum of squared proportions × (q / (q-1)) where q is the number of categories.
    cats = set(r1) | set(r2)
    q = len(cats)
    if q < 2:
        return 1.0  # all the same — perfect agreement, chance is also perfect
    pe = 0.0
    for c in cats:
        p_c = (r1.count(c) + r2.count(c)) / (2 * n)
        pe += p_c * (1 - p_c)
    pe = pe / (q - 1)
    if pe >= 1:
        return float("nan")
    return (po - pe) / (1 - pe)


ac1_binary = gwets_ac1(extract_a1_bin, extract_a2_bin)
ac1_3class = gwets_ac1(extract_a1_3c, extract_a2_3c)


# POS agreement (when both extracted and both have POS)
pos_compare = both_extracted_df[
    (both_extracted_df["pos_a1"] != "") & (both_extracted_df["pos_a2"] != "")
]
pos_agree = (pos_compare["pos_a1"] == pos_compare["pos_a2"]).sum()
pos_agreement_rate = pos_agree / len(pos_compare) if len(pos_compare) else float("nan")


def _round(x):
    return round(x, 4) if not pd.isna(x) else "n/a"


iaa_rows = [
    # Outcome counts
    {"metric": "total_lemmas",                  "value": n_total},
    {"metric": "agreed_extracted_match",        "value": n_agreed_extracted},
    {"metric": "agreed_no_pair",                "value": n_agreed_no_pair},
    {"metric": "agreed_not_found",              "value": n_agreed_not_found},
    {"metric": "agreed_empty_other",            "value": n_agreed_empty_other},
    {"metric": "disagree_content",              "value": n_disagree_content},
    {"metric": "disagree_extract",              "value": n_disagree_extract},
    {"metric": "unclear",                       "value": n_unclear},
    # Agreement rates
    {"metric": "row_agreement_rate",            "value": _round(row_agreement_rate)},
    {"metric": "percent_agreement_extract",     "value": _round(percent_agreement_binary)},
    {"metric": "cond_content_agreement",        "value": _round(cond_content_agreement)},
    {"metric": "content_pct_agreement",         "value": _round(content_pct_agreement)},
    # κ variants
    {"metric": "cohen_kappa_extract_binary",    "value": _round(kappa_binary)},
    {"metric": "cohen_kappa_extract_3class",    "value": _round(kappa_3class)},
    # Gwet's AC1
    {"metric": "gwet_ac1_extract_binary",       "value": _round(ac1_binary)},
    {"metric": "gwet_ac1_extract_3class",       "value": _round(ac1_3class)},
    # POS
    {"metric": "pos_agreement_rate",            "value": _round(pos_agreement_rate)},
    {"metric": "pos_compared_rows",             "value": len(pos_compare)},
]
iaa_df = pd.DataFrame(iaa_rows)
iaa_path = OUT_DIR / f"{DICT_ID}_iaa_report.csv"
iaa_df.to_csv(iaa_path, index=False)

print("=== IAA report ===")
print(iaa_df.to_string(index=False))
print(f"\nWritten: {iaa_path.name}")

=== IAA report ===
                    metric    value
              total_lemmas 100.0000
    agreed_extracted_match  88.0000
            agreed_no_pair   3.0000
          agreed_not_found   0.0000
        agreed_empty_other   0.0000
          disagree_content   9.0000
          disagree_extract   0.0000
                   unclear   0.0000
        row_agreement_rate   0.9100
 percent_agreement_extract   1.0000
    cond_content_agreement   0.9072
     content_pct_agreement   0.9072
cohen_kappa_extract_binary   1.0000
cohen_kappa_extract_3class   1.0000
   gwet_ac1_extract_binary   1.0000
   gwet_ac1_extract_3class   1.0000
        pos_agreement_rate   1.0000
         pos_compared_rows  96.0000

Written: 91_iaa_report.csv


## 7. Stage 2 — Load legacy parcor (pipeline A) and align to gold

For each `source_lemma` in our 100-row sample (not just C — we want to evaluate
recall across *all* sampled lemmas, including ones where C is empty), look up
the legacy pipeline's output:

In [460]:
pipeline_df = pd.read_csv(PARCOR_PATH)
print(f"Pipeline parcor: {len(pipeline_df)} total rows, columns={pipeline_df.columns.tolist()[:8]}...")

# Normalize kalimat columns (the only ones we need for content-keyed comparison).
# Note: spellchecked parcor files do not have a `lemma` column, so we cannot
# do lemma-keyed lookup. We use content-keyed comparison instead: for each
# gold pair, we check whether any pipeline row contains the same pair
# (normalized asal AND tujuan from the same pipeline row).
#
# Also strip CR/LF and collapse whitespace to neutralize CSV-cell line-break
# artifacts that would otherwise fail normalized matching.
for col in ["kalimat_asal", "kalimat_tujuan", "lemma"]:
    if col in pipeline_df.columns:
        pipeline_df[col] = (pipeline_df[col].fillna("").astype(str)
                            .str.replace("\r", " ", regex=False)
                            .str.replace("\n", " ", regex=False)
                            .str.replace(r"\s+", " ", regex=True)
                            .str.strip())

# Precompute normalized (asal, tuj) pairs for fast TP lookup
pipeline_normalized_pairs = set()
for _, row in pipeline_df.iterrows():
    asal = row.get("kalimat_asal", "")
    tuj  = row.get("kalimat_tujuan", "")
    if asal and tuj:
        pipeline_normalized_pairs.add((normalize_text(asal), normalize_text(tuj)))

# For strict and fuzzy match levels, keep the list of raw pairs
pipeline_pair_list = []
for _, row in pipeline_df.iterrows():
    asal = row.get("kalimat_asal", "")
    tuj  = row.get("kalimat_tujuan", "")
    if asal and tuj:
        pipeline_pair_list.append((asal, tuj))

# Lemma → row lookup (only used for POS comparison in §10, not for parcor comparison)
# Only available when the pipeline file has a `lemma` column (audit-stage Fixed files)
pipeline_lemma_lookup = {}
if "lemma" in pipeline_df.columns:
    for _, row in pipeline_df.iterrows():
        lem = row["lemma"]
        if lem and lem not in pipeline_lemma_lookup:
            pipeline_lemma_lookup[lem] = {
                "kalimat_asal":   row.get("kalimat_asal", ""),
                "kalimat_tujuan": row.get("kalimat_tujuan", ""),
                "main_lemma":     row.get("main_lemma", ""),
            }

print(f"Distinct (asal, tuj) normalized pairs in pipeline: {len(pipeline_normalized_pairs)}")
print(f"Raw pipeline pairs (for fuzzy match): {len(pipeline_pair_list)}")
print(f"Pipeline has lemma column for POS lookup: {bool(pipeline_lemma_lookup)}")


Pipeline parcor: 4773 total rows, columns=['kalimat_asal_original', 'kalimat_tujuan_original', 'kalimat_asal', 'kalimat_tujuan', 'lemma', 'main_lemma', 'placeholders_expanded_asal', 'placeholders_expanded_tujuan']...
Distinct (asal, tuj) normalized pairs in pipeline: 4711
Raw pipeline pairs (for fuzzy match): 4771
Pipeline has lemma column for POS lookup: True


## 8. Stage 2 (cont.) Row-level comparison

For each sampled lemma, classify the C-vs-A outcome.

**Important:** C is the strict-intersection reconciled gold, which includes
*only* `agreed_extracted_match` rows. For evaluation, we expand C to also
recognize `agreed_no_pair` and `agreed_not_found` as gold "empty" cases, 
these are also gold-valid judgments, just that the gold says "no pair to
extract here."

Lemmas with `disagree_*` or `unclear` outcomes are **excluded from the A-vs-C
comparison** because there's no gold answer for them yet (researcher needs to
adjudicate). They're reported in a separate count.

In [461]:
def detect_concatenation(a_text: str, c_text: str) -> bool:
    """Heuristic: A's content is likely a concatenation of multiple sentences."""
    if not a_text:
        return False
    sentence_ends = sum(a_text.count(c) for c in ".!?")
    if sentence_ends >= CONCAT_MIN_SENTENCE_ENDS:
        return True
    if c_text and len(a_text) > CONCAT_LENGTH_RATIO * len(c_text):
        return True
    return False


def find_matching_pair(c_asal: str, c_tuj: str):
    """Find a pipeline pair matching (c_asal, c_tuj) and report match levels.
    
    Returns dict with match info. The match-quality bits are computed against
    the BEST pipeline pair (in case multiple pipeline rows partially match).
    
    Match logic:
    - strict: some pipeline pair (raw) equals (c_asal, c_tuj) exactly
    - normalized: some pipeline pair, after normalize_text, equals (norm(c_asal), norm(c_tuj))
    - fuzzy: some pipeline pair has fuzzy_similarity >= FUZZY_THRESHOLD on BOTH sides
    """
    if not c_asal or not c_tuj:
        return {
            "found_strict": False, "found_normalized": False, "found_fuzzy": False,
            "best_a_asal": "", "best_a_tuj": "",
            "best_asal_fuzzy_score": 0.0, "best_tuj_fuzzy_score": 0.0,
        }
    
    # Strict and normalized are fast: hash lookup
    c_norm = (normalize_text(c_asal), normalize_text(c_tuj))
    found_normalized = c_norm in pipeline_normalized_pairs
    
    # Strict: scan raw pairs (cheap)
    found_strict = any(a == c_asal and b == c_tuj for a, b in pipeline_pair_list)
    
    # Fuzzy: scan all pairs, find the best
    best_score = 0.0
    best_pair = ("", "")
    best_asal_score = 0.0
    best_tuj_score = 0.0
    for a_asal, a_tuj in pipeline_pair_list:
        s_asal = fuzzy_similarity(c_asal, a_asal)
        s_tuj  = fuzzy_similarity(c_tuj, a_tuj)
        combined = min(s_asal, s_tuj)  # both sides must pass; use min
        if combined > best_score:
            best_score = combined
            best_pair = (a_asal, a_tuj)
            best_asal_score = s_asal
            best_tuj_score = s_tuj
    found_fuzzy = best_score >= FUZZY_THRESHOLD
    
    return {
        "found_strict":         found_strict,
        "found_normalized":     found_normalized,
        "found_fuzzy":          found_fuzzy,
        "best_a_asal":          best_pair[0],
        "best_a_tuj":           best_pair[1],
        "best_asal_fuzzy_score": round(best_asal_score, 3),
        "best_tuj_fuzzy_score":  round(best_tuj_score, 3),
    }


comparison_rows = []
excluded_count = 0

for _, rec in reconcile_df.iterrows():
    lemma = rec["source_lemma"]
    outcome_iaa = rec["outcome"]

    # Determine gold (C) status
    if outcome_iaa == "agreed_extracted_match":
        c_has_pair = True
        c_asal = rec["kalimat_asal_a1"]
        c_tuj  = rec["kalimat_tujuan_a1"]
    elif outcome_iaa in ("agreed_no_pair", "agreed_not_found", "agreed_empty_other"):
        c_has_pair = False
        c_asal = ""
        c_tuj  = ""
    else:
        # Disagreement — exclude from A-vs-C
        excluded_count += 1
        continue

    # Content-keyed lookup: is gold pair found in pipeline?
    if c_has_pair:
        match = find_matching_pair(c_asal, c_tuj)
        a_asal = match["best_a_asal"]  # best fuzzy match (for inspection)
        a_tuj  = match["best_a_tuj"]
        is_concat = detect_concatenation(a_asal, c_asal) or detect_concatenation(a_tuj, c_tuj)
    else:
        match = {"found_strict": False, "found_normalized": False, "found_fuzzy": False,
                 "best_a_asal": "", "best_a_tuj": "",
                 "best_asal_fuzzy_score": 0.0, "best_tuj_fuzzy_score": 0.0}
        a_asal = ""
        a_tuj  = ""
        is_concat = False

    # Classify outcome under content-keyed comparison (primary metric: normalized)
    if c_has_pair and match["found_normalized"]:
        outcome = "TP"
    elif c_has_pair and not match["found_normalized"]:
        outcome = "FN"  # gold pair not found in pipeline at the normalized level
    else:
        # c_has_pair is False — gold has no pair to extract
        outcome = "TN"

    comparison_rows.append({
        "source_lemma":              lemma,
        "iaa_outcome":               outcome_iaa,
        "c_has_pair":                c_has_pair,
        "c_kalimat_asal":            c_asal,
        "c_kalimat_tujuan":          c_tuj,
        "best_a_kalimat_asal":       a_asal,   # for inspection only; not the only candidate
        "best_a_kalimat_tujuan":     a_tuj,
        "match_strict":              match["found_strict"],
        "match_normalized":          match["found_normalized"],
        "match_fuzzy":               match["found_fuzzy"],
        "best_asal_fuzzy_score":     match["best_asal_fuzzy_score"],
        "best_tuj_fuzzy_score":      match["best_tuj_fuzzy_score"],
        "concatenation_candidate":   is_concat,
        "outcome":                   outcome,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = OUT_DIR / f"{DICT_ID}_comparison_A_vs_C.csv"
comparison_df.to_csv(comparison_path, index=False)

print(f"Comparison: {len(comparison_df)} rows evaluated, {excluded_count} excluded (disagreement)")
print(f"Written: {comparison_path.name}")

# Outcome breakdown
print("\n=== A-vs-C outcomes (content-keyed) ===")
for outcome, n in Counter(comparison_df["outcome"]).most_common():
    print(f"  {outcome}: {n}")
print(f"  concatenation_candidate flagged: {comparison_df['concatenation_candidate'].sum()}")


Comparison: 91 rows evaluated, 9 excluded (disagreement)
Written: 91_comparison_A_vs_C.csv

=== A-vs-C outcomes (content-keyed) ===
  TP: 52
  FN: 36
  TN: 3
  concatenation_candidate flagged: 0


## 9. Compute precision / recall / F1 at each match level

Three metric variants depending on what counts as a TP:

- **Strict**: TP only if `outcome == "TP"` *and* `match_asal_strict` *and* `match_tujuan_strict`
- **Normalized**: TP if `outcome == "TP"` (which already uses normalized match)
- **Fuzzy**: TP if both filled and fuzzy match passes both sides

In [462]:
def compute_metrics(df):
    """Under content-keyed comparison: only recall is well-defined per level.
    
    For each match level (strict/normalized/fuzzy):
    - TP = c_has_pair AND match_<level>
    - FN = c_has_pair AND NOT match_<level>
    - TN = NOT c_has_pair
    - FP is undefined (no per-lemma expectation; collapsed into FN)
    
    Recall = TP / (TP + FN); precision and F1 are NaN.
    """
    metrics = {}
    for level in ["strict", "normalized", "fuzzy"]:
        tp = fn = tn = 0
        for _, row in df.iterrows():
            c_has = row["c_has_pair"]
            m = row[f"match_{level}"]
            if c_has and m:
                tp += 1
            elif c_has and not m:
                fn += 1
            else:
                tn += 1
        recall = tp / (tp + fn) if (tp + fn) else float("nan")
        metrics[level] = {
            "tp":         tp,
            "fn":         fn,
            "tn":         tn,
            "fp":         float("nan"),  # undefined under content-keyed
            "precision":  float("nan"),  # undefined under content-keyed
            "recall":     round(recall, 4) if not pd.isna(recall) else float("nan"),
            "f1":         float("nan"),  # undefined under content-keyed
        }
    return metrics


metrics = compute_metrics(comparison_df)
summary_rows = []
for level, m in metrics.items():
    summary_rows.append({"match_level": level, **m})
summary_df = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / f"{DICT_ID}_comparison_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("=== Pipeline A vs Gold C — summary (content-keyed) ===")
print(summary_df.to_string(index=False))
print()
print("Note: under content-keyed comparison, precision and F1 are undefined")
print("because we cannot define 'false positives' without a per-lemma expectation.")
print("Recall is the primary metric.")
print(f"\nWritten: {summary_path.name}")


=== Pipeline A vs Gold C — summary (content-keyed) ===
match_level  tp  fn  tn  fp  precision  recall  f1
     strict  40  48   3 NaN        NaN  0.4545 NaN
 normalized  52  36   3 NaN        NaN  0.5909 NaN
      fuzzy  70  18   3 NaN        NaN  0.7955 NaN

Note: under content-keyed comparison, precision and F1 are undefined
because we cannot define 'false positives' without a per-lemma expectation.
Recall is the primary metric.

Written: 91_comparison_summary.csv


## 10. Stage 3 POS comparison with morphology file

The morphology file uses schema: `kata` (main lemma), `form` (derived form), `tag` (POS).
We look up each `source_lemma` against `form` first (since most sampled lemmas
are derived forms like `pengacara`, `kearsipan`), and fall back to `kata` if no
form match exists.

Coverage is expected to be partial, the morphology file may not have entries
for every lemma. Missing morphology is not a pipeline failure; it's logged
separately.

In [463]:
# Find morphology file
morph_path = None
for cand in MORPH_CANDIDATES:
    if cand.exists():
        morph_path = cand
        break

if morph_path is None:
    print("⚠ Morphology file not found at any candidate path:")
    for c in MORPH_CANDIDATES:
        print(f"   {c}")
    print("Skipping POS-vs-morphology comparison.")
    morph_lookup = {}
else:
    morph_df = pd.read_csv(morph_path)
    print(f"Loaded morphology: {len(morph_df)} rows from {morph_path.name}")
    for col in ["kata", "form", "tag"]:
        if col in morph_df.columns:
            morph_df[col] = morph_df[col].fillna("").astype(str).str.strip()
    # Build form → tag lookup (first occurrence wins)
    form_lookup = {}
    for _, row in morph_df.iterrows():
        if row["form"] and row["form"] not in form_lookup:
            form_lookup[row["form"]] = row["tag"]
    # Build kata → tag fallback (first occurrence)
    kata_lookup = {}
    for _, row in morph_df.iterrows():
        if row["kata"] and row["kata"] not in kata_lookup:
            kata_lookup[row["kata"]] = row["tag"]
    morph_lookup = {"form": form_lookup, "kata": kata_lookup}
    print(f"  Unique `form` entries: {len(form_lookup)}")
    print(f"  Unique `kata` entries: {len(kata_lookup)}")


def lookup_morph_pos(lemma: str):
    """Return (pos, source) where source is 'form', 'kata', or 'missing'."""
    if not morph_lookup:
        return ("", "missing")
    if lemma in morph_lookup.get("form", {}):
        return (morph_lookup["form"][lemma], "form")
    if lemma in morph_lookup.get("kata", {}):
        return (morph_lookup["kata"][lemma], "kata")
    return ("", "missing")


# Compare each row in parcor C against morphology
pos_rows = []
for _, rec in reconcile_df.iterrows():
    lemma = rec["source_lemma"]
    pos_a1 = rec["pos_a1"]
    pos_a2 = rec["pos_a2"]
    morph_pos, morph_source = lookup_morph_pos(lemma)
    pos_rows.append({
        "source_lemma":      lemma,
        "iaa_outcome":       rec["outcome"],
        "pos_a1":            pos_a1,
        "pos_a2":            pos_a2,
        "pos_annotator_agree": pos_a1 == pos_a2 and pos_a1 != "",
        "morph_pos":         morph_pos,
        "morph_source":      morph_source,
        "morph_matches_a1":  morph_pos == pos_a1 and morph_pos != "",
        "morph_matches_a2":  morph_pos == pos_a2 and morph_pos != "",
    })

pos_df = pd.DataFrame(pos_rows)
pos_path = OUT_DIR / f"{DICT_ID}_pos_comparison.csv"
pos_df.to_csv(pos_path, index=False)

# Summary stats
both_extracted_mask = pos_df["iaa_outcome"].isin(["agreed_extracted_match", "disagree_content"])
extracted_pos = pos_df[both_extracted_mask & (pos_df["pos_a1"] != "")]

print("=== POS comparison ===")
print(f"Lemmas with annotator POS (both extracted, A1 has POS): {len(extracted_pos)}")
if len(extracted_pos):
    print(f"  A1 == A2: {extracted_pos['pos_annotator_agree'].sum()} / {len(extracted_pos)} "
          f"({extracted_pos['pos_annotator_agree'].mean():.1%})")
    morph_covered = extracted_pos[extracted_pos["morph_source"] != "missing"]
    print(f"\n  Morphology coverage: {len(morph_covered)} / {len(extracted_pos)} "
          f"({len(morph_covered) / len(extracted_pos):.1%})")
    if len(morph_covered):
        print(f"     via form lookup: {(morph_covered['morph_source'] == 'form').sum()}")
        print(f"     via kata fallback: {(morph_covered['morph_source'] == 'kata').sum()}")
        print(f"  Morph POS == A1 POS: {morph_covered['morph_matches_a1'].sum()} / {len(morph_covered)}")

print(f"\nWritten: {pos_path.name}")

Loaded morphology: 3545 rows from 91_Morphology.csv
  Unique `form` entries: 3454
  Unique `kata` entries: 1567
=== POS comparison ===
Lemmas with annotator POS (both extracted, A1 has POS): 96
  A1 == A2: 96 / 96 (100.0%)

  Morphology coverage: 26 / 96 (27.1%)
     via form lookup: 15
     via kata fallback: 11
  Morph POS == A1 POS: 17 / 26

Written: 91_pos_comparison.csv


## 11. Final summary

In [464]:
print("=" * 60)
print(f"  Gold Annotation Analysis — Dict #{DICT_ID}")
print("=" * 60)
print(f"\n[IAA — A1 vs A2]")
print(f"  Total sampled lemmas: {n_total}")
print(f"  Agreed (matched extraction): {n_agreed_extracted}")
print(f"  Agreed (no pair):            {n_agreed_no_pair}")
print(f"  Agreed (not found):          {n_agreed_not_found}")
print(f"  Agreed (empty other):        {n_agreed_empty_other}")
print(f"  Disagreement (content):      {n_disagree_content}")
print(f"  Disagreement (extract):      {n_disagree_extract}")
print(f"  Unclear:                     {n_unclear}")
print()
print(f"  Raw agreement rate:                  {row_agreement_rate:.4f}")
print(f"  Percent agreement (extract/skip):    {percent_agreement_binary:.4f}")
print(f"  Conditional content agreement:       {cond_content_agreement:.4f}")
print()
print(f"  Cohen's κ (extract/skip binary):     {kappa_binary:.4f}")
print(f"  Cohen's κ (extract/skip 3-class):    {kappa_3class:.4f}")
print(f"  Gwet's AC1 (extract/skip binary):    {ac1_binary:.4f}")
print(f"  Gwet's AC1 (extract/skip 3-class):   {ac1_3class:.4f}")

print(f"\n[Parcor C → Pipeline A comparison]")
print(f"  Evaluable lemmas (non-disagreement): {len(comparison_df)}")
print(f"  Concatenation candidates:            {comparison_df['concatenation_candidate'].sum()}")
print(f"\n  Metrics at each match level:")
for level, m in metrics.items():
    print(f"    {level:<12} P={m['precision']}, R={m['recall']}, F1={m['f1']}  "
          f"(TP={m['tp']}, FP={m['fp']}, FN={m['fn']}, TN={m['tn']})")

print(f"\n[Output files in {OUT_DIR.name}/]")
for p in sorted(OUT_DIR.glob(f"{DICT_ID}_*.csv")):
    print(f"  {p.name}")

  Gold Annotation Analysis — Dict #91

[IAA — A1 vs A2]
  Total sampled lemmas: 100
  Agreed (matched extraction): 88
  Agreed (no pair):            3
  Agreed (not found):          0
  Agreed (empty other):        0
  Disagreement (content):      9
  Disagreement (extract):      0
  Unclear:                     0

  Raw agreement rate:                  0.9100
  Percent agreement (extract/skip):    1.0000
  Conditional content agreement:       0.9072

  Cohen's κ (extract/skip binary):     1.0000
  Cohen's κ (extract/skip 3-class):    1.0000
  Gwet's AC1 (extract/skip binary):    1.0000
  Gwet's AC1 (extract/skip 3-class):   1.0000

[Parcor C → Pipeline A comparison]
  Evaluable lemmas (non-disagreement): 91
  Concatenation candidates:            0

  Metrics at each match level:
    strict       P=nan, R=0.4545, F1=nan  (TP=40, FP=nan, FN=48, TN=3)
    normalized   P=nan, R=0.5909, F1=nan  (TP=52, FP=nan, FN=36, TN=3)
    fuzzy        P=nan, R=0.7955, F1=nan  (TP=70, FP=nan, FN=18, TN